In [35]:
# imports and parameter
import glob
from pathlib import Path

In [36]:
# function to read in the cuda mean miner results to get all cycles found
def read_verify(filename):
    last_nonce = -1
    verified = 0
    nonces = 0
    solutions = []
    with open(filename) as file:
        for line in file:
            data = line.split()
            if(data[0] == 'nonce'):
                last_nonce = int(data[1])
                if last_nonce != nonces:
                    print("warning!!!!: %i" % nonces)
                nonces += 1
            if(data[0] == 'Solution'):
                solutions.append((last_nonce,[int(x,16) for x in data[1:]]))
            if(data[0] == 'Verified'):
                verified += 1
    if verified != len(solutions):
        print('Warning: Not all solutions were verified (%i out of %i) in file "%s"!' % (verified, len(solutions), filename))
    return solutions, nonces

In [37]:
def validate(bits, logk, batches):
    print("===========================================================================================")
    print("Results for Cuckatoo%i using 2**%i edges per batch and %i batches:" % (bits, logk, batches))
    print("")
    results = [(f,read_verify(f)) for f in glob.glob("data/verify_mean%icu_*.out" % bits)]
    agg_nonces = 0
    agg_solutions = 0
    agg_hits = 0
    agg_hitlist = {}
    agg_attempts = 0
    for (f, (solutions, nonces)) in results:
        print('File "%s": %i solutions found in %i attempts, which gives on average %.2f attempts before success.' % (f, len(solutions), nonces, nonces/len(solutions)))        
        agg_nonces += nonces
        agg_solutions += len(solutions)
        hitlist = []
        for (act_nonce, edges) in solutions:
            if edges[0] < (2**(bits-logk)*batches):
                hitlist.append(act_nonce)
        print('%i attempts needed to find %i cycles, which gives on average %.3f attempts before success or %.3fx more attempts needed than using mean solver.' % (hitlist[-1], len(hitlist), hitlist[-1]/len(hitlist), hitlist[-1]/len(hitlist)/42))
        print("Hits: ", hitlist)
        print('')
        agg_attempts += hitlist[-1]
        agg_hits += len(hitlist)
        agg_hitlist.update({ f : hitlist })
    print('In total: %i solutions found in %i attempts, which gives on average %.3f attempts before success.' % (agg_solutions, agg_nonces, agg_nonces/agg_solutions))
    print('In total: %i hits found in %i attempts, which gives on average %.3fx attempts before success, and %.3f more attempts needed than mean solver.' % (agg_hits, agg_attempts, agg_attempts/agg_hits, agg_attempts/agg_hits/42))
    print('')
    return agg_hitlist

In [38]:
bits = [27, 29, 31]
kfacts = [10, 11, 12, 13, 14, 15]
hits = {}
for b in bits:
    for k in kfacts:
        hits.update({ (b,k,4) : validate(b, k, 4) })

Results for Cuckatoo27 using 2**10 edges per batch and 4 batches:

File "data/verify_mean27cu_seed1.out": 554 solutions found in 24000 attempts, which gives on average 43.32 attempts before success.
23826 attempts needed to find 91 cycles, which gives on average 261.824 attempts before success or 6.234x more attempts needed than using mean solver.
Hits:  [187, 1116, 1560, 1616, 1627, 1915, 1981, 2329, 2563, 2889, 3062, 3276, 3676, 3958, 3976, 4333, 4346, 4378, 4674, 5047, 5344, 5442, 5795, 6043, 6311, 6589, 6943, 7632, 7710, 7779, 8138, 8644, 8805, 8829, 9718, 10812, 10893, 11419, 11797, 11846, 12097, 12591, 12599, 13305, 13804, 14030, 14374, 14548, 14728, 15707, 15781, 15885, 16185, 16253, 16678, 16785, 17310, 17343, 17401, 17482, 17514, 18225, 18248, 18791, 19127, 19277, 19364, 19372, 19517, 19723, 20489, 20513, 20611, 20637, 20752, 20784, 21254, 21686, 21802, 21874, 22216, 22467, 22470, 22849, 22968, 23221, 23231, 23307, 23530, 23737, 23826]

File "data/verify_mean27cu_seed0.out": 5

In [41]:
seed0 = "000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000"
seed1 = "000005EED100000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000"
seed2 = "00000000005EED2000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000"
bits = [27, 29, 31]
for b in bits:
    with open("verify_splean%ix8_k14_t8_seed0.sh" % b, "w") as file:
        for h in hits[(b,14,4)]['data/verify_mean%icu_seed0.out'%b]:
            file.write("./splean%ix8_k14_t8 -n %i -x %s >> verify_splean%ix8_k14_t8_seed0.out \n" % (b, h, seed0, b))
    with open("verify_splean%ix8_k14_t8_seed1.sh" % b, "w") as file:
        for h in hits[(b,14,4)]['data/verify_mean%icu_seed1.out'%b]:
            file.write("./splean%ix8_k14_t8 -n %i -x %s >> verify_splean%ix8_k14_t8_seed1.out \n" % (b, h, seed1, b))
    with open("verify_splean%ix8_k14_t8_seed2.sh" % b, "w") as file:
        for h in hits[(b,14,4)]['data/verify_mean%icu_seed2.out'%b]:
            file.write("./splean%ix8_k14_t8 -n %i -x %s >> verify_splean%ix8_k14_t8_seed2.out \n" % (b, h, seed2, b))